In [0]:
%sql
---- Creating new catalog, schema -----
create catalog if not exists sql_youtube_practise;
use catalog sql_youtube_practise;
create schema if not exists sql;
use sql;
show current schema;

catalog,namespace
sql_youtube_practise,sql


##### Pareto Analysis (80/20 Rule)

Concept:
- The Pareto Principle (80/20 Rule) states that approximately 80% of the outcome comes from 20% of the causes.
In business, this often means a small number of products generate most of the revenue.

Question:
- Find the products contributing to the first 80% of total sales.

Idea:
- Aggregate sales by product.
- Calculate cumulative sales.
- Keep products until cumulative sales reaches 80% of total sales.

Concepts:
GROUP BY | Window Function | Running Total | CTE

In [0]:
%sql
with product_sales as (
    select
        Product_ID,
        sum(sales) as total_sales
    from superstore_orders
    group by Product_ID
),
running_total as (
    select
        Product_ID,
        sum(total_sales) over (order by total_sales desc, Product_ID rows between unbounded preceding and current row) as cumulative_sales,
        0.8*sum(total_sales) over () as ultimate_sales
    from product_sales
)
select
    Product_ID,
    cumulative_sales,
    ultimate_sales
from running_total
where cumulative_sales <= ultimate_sales;

Product_ID,cumulative_sales,ultimate_sales
TEC-CO-10004722,61599.824,1837760.6882399973
OFF-BI-10003527,89053.208,1837760.6882399973
TEC-MA-10002412,111691.688,1837760.6882399973
FUR-CH-10002024,133562.264,1837760.6882399973
OFF-BI-10001359,153385.743,1837760.6882399973
OFF-BI-10000545,172410.243,1837760.6882399973
TEC-CO-10001449,191249.929,1837760.6882399973
TEC-MA-10001127,209624.824,1837760.6882399973
OFF-BI-10004995,227589.892,1837760.6882399973
OFF-SU-10000151,244620.204,1837760.6882399973


##### 01. Analyze customers' orders and products purchased to identify relationships or purchasing patterns.

In [0]:
%sql
DROP TABLE IF EXISTS customer_orders;
DROP TABLE IF EXISTS customer_products;
CREATE TABLE customer_orders (order_id INT, customer_id INT, product_id INT);
INSERT INTO customer_orders VALUES
(1, 1, 1),
(1, 1, 2),
(1, 1, 3),
(2, 2, 1),
(2, 2, 2),
(2, 2, 4),
(3, 1, 5);
CREATE TABLE customer_products (id INT, name STRING);
INSERT INTO customer_products VALUES
(1, 'A'),
(2, 'B'),
(3, 'C'),
(4, 'D'),
(5, 'E');
SELECT * FROM customer_orders;

order_id,customer_id,product_id
1,1,1
1,1,2
1,1,3
2,2,1
2,2,2
2,2,4
3,1,5


In [0]:
%sql
SELECT * FROM customer_products;

id,name
1,A
2,B
3,C
4,D
5,E


In [0]:
%sql
with cte as (
select 
    o1.product_id as a, 
    o2.product_id as b,
    count(*) as freq
from customer_orders as o1
    inner join
customer_orders as o2
on o1.order_id = o2.order_id
where o1.product_id < o2.product_id
group by o1.product_id, o2.product_id
)

select 
    concat_ws(' ', c1.name, c2.name) as products,
    freq
from cte
join customer_products as c1
on cte.a = c1.id
join customer_products as c2
on cte.b = c2.id;

products,freq
A C,1
A D,1
B C,1
A B,2
B D,1


##### 02. Given the 2 tables, return the fraction of users, rounded to 2 decimal places, who accessed Amazon Music and upgraded to Prime Membership within the first 30 days of signing up.

In [0]:
%sql
DROP TABLE IF EXISTS app_users;
DROP TABLE IF EXISTS app_events;

CREATE TABLE app_users (
    user_id INT,
    name STRING,
    join_date DATE
);

INSERT INTO app_users VALUES
(1, 'Jon', DATE('2020-02-14')),
(2, 'Jane', DATE('2020-02-14')),
(3, 'Jill', DATE('2020-02-15')),
(4, 'Josh', DATE('2020-02-15')),
(5, 'Jean', DATE('2020-02-16')),
(6, 'Justin', DATE('2020-02-17')),
(7, 'Jeremy', DATE('2020-02-18'));

CREATE TABLE app_events (
    user_id INT,
    type STRING,
    access_date DATE
);

INSERT INTO app_events VALUES
(1, 'Pay',   DATE('2020-03-01')),
(2, 'Music', DATE('2020-03-02')),
(2, 'P',     DATE('2020-03-12')),
(3, 'Music', DATE('2020-03-15')),
(4, 'Music', DATE('2020-03-15')),
(1, 'P',     DATE('2020-03-16')),
(3, 'P',     DATE('2020-03-22'));

SELECT * FROM app_users;

user_id,name,join_date
1,Jon,2020-02-14
2,Jane,2020-02-14
3,Jill,2020-02-15
4,Josh,2020-02-15
5,Jean,2020-02-16
6,Justin,2020-02-17
7,Jeremy,2020-02-18


In [0]:
%sql
SELECT * FROM app_events;

user_id,type,access_date
1,Pay,2020-03-01
2,Music,2020-03-02
2,P,2020-03-12
3,Music,2020-03-15
4,Music,2020-03-15
1,P,2020-03-16
3,P,2020-03-22


In [0]:
%sql
with cte as (
select
    u.*,
    e.type,
    e.access_date,
    date_diff(day, u.join_date, e.access_date) as no_of_days
from app_users as u
left join app_events as e
on u.user_id = e.user_id and e.type = "P"
where u.user_id in (
    select user_id from app_events where type = "Music")
)
select
    count(distinct(user_id)) as total_no_of_users,
    sum(
        case
            when no_of_days <= 30 then 1
            else 0
        end
    ) as users_within_30_days,
    round((sum(case when no_of_days <= 30 then 1 end))/(count(distinct(user_id)))*100,2) as conversion_rate
from cte;

total_no_of_users,users_within_30_days,conversion_rate
3,1,33.33


#### 03. Calculate customer retention and churn metrics based on customers' purchase history.
###### Identify recurring (retained) customers and churned customers based on their previous and next orders.

In [0]:
%sql
DROP TABLE IF EXISTS customer_transactions;
CREATE TABLE customer_transactions (
    order_id INT,
    cust_id INT,
    order_date DATE,
    amount INT
);

INSERT INTO customer_transactions VALUES
(1, 1, '2020-01-15', 150),
(2, 1, '2020-02-10', 150),
(3, 2, '2020-01-16', 150),
(4, 2, '2020-02-25', 150),
(5, 3, '2020-01-10', 150),
(6, 3, '2020-02-20', 150),
(7, 4, '2020-01-20', 150),
(8, 5, '2020-02-20', 150);

SELECT * FROM customer_transactions ORDER BY order_id;

order_id,cust_id,order_date,amount
1,1,2020-01-15,150
2,1,2020-02-10,150
3,2,2020-01-16,150
4,2,2020-02-25,150
5,3,2020-01-10,150
6,3,2020-02-20,150
7,4,2020-01-20,150
8,5,2020-02-20,150


In [0]:
%sql
with cte as (
    select
        *,
        lag(order_date) over(partition by cust_id order by order_date) as prev_order_date,
        lead(order_date) over(partition by cust_id order by order_date) as next_order_date
    from customer_transactions
)
select
    extract(month from order_date) as order_month,
    sum(case
            when (extract(month from order_date))-(extract(month from prev_order_date)) = 1 then 1
            else 0
        end) as recurring,
    sum(case
            when (prev_order_date is null) and (next_order_date is null) then 1
            else 0
    end) as churned,
    count(*) as no_of_customers
from cte
group by extract(month from order_date);


order_month,recurring,churned,no_of_customers
1,0,1,4
2,3,1,4


#### 04. Find the second most recent activity for each user; if a user has only one activity, return that activity.

In [0]:
%sql
DROP TABLE IF EXISTS customer_activity;
CREATE TABLE customer_activity (
    username VARCHAR(20),
    activity VARCHAR(20),
    startDate DATE,
    endDate DATE
);
INSERT INTO customer_activity VALUES
('Alice', 'Travel',  '2020-02-12', '2020-02-20'),
('Alice', 'Dancing', '2020-02-21', '2020-02-23'),
('Alice', 'Travel',  '2020-02-24', '2020-02-28'),
('Bob',   'Travel',  '2020-02-11', '2020-02-18');
SELECT * FROM customer_activity;

username,activity,startDate,endDate
Alice,Travel,2020-02-12,2020-02-20
Alice,Dancing,2020-02-21,2020-02-23
Alice,Travel,2020-02-24,2020-02-28
Bob,Travel,2020-02-11,2020-02-18


In [0]:
%sql
-------------------------------------------  Workaround 1  ---------------------------------------------
with cte as(
    select
        *,
        rank() over(partition by username order by endDate desc) as rank,
        count(*) over(partition by username) as total_users
    from customer_activity
)
select
    *
from cte where rank = 2 or total_users = 1;


-------------------------------------------- Workaround 2 ----------------------------------------------
-- with cte as (
--     select *
--     from customer_activity where username in (
--         select username from customer_activity group by username having count(*) = 1
--     )
-- ),
-- cte1 as (
--         select 
--             *, 
--             rank() over(partition by username order by endDate desc) as rank
--         from customer_activity
-- )
-- select * from cte
-- union
-- select username, activity, startDate, endDate from cte1 where rank = 2;

username,activity,startDate,endDate,rank,total_users
Alice,Dancing,2020-02-21,2020-02-23,2,3
Bob,Travel,2020-02-11,2020-02-18,1,1


username,activity,startDate,endDate
Bob,Travel,2020-02-11,2020-02-18
Alice,Dancing,2020-02-21,2020-02-23


##### 05. Calculate each employee's total bill by applying the billing rate that was effective on each employee's work date.

In [0]:
%sql
DROP TABLE IF EXISTS employee_billings;
DROP TABLE IF EXISTS employee_hoursworked;
CREATE TABLE employee_billings (
    emp_name VARCHAR(10),
    bill_date DATE,
    bill_rate INT
);
INSERT INTO employee_billings VALUES
('Sachin', '1990-01-01', 25),
('Sehwag', '1989-01-01', 15),
('Dhoni', '1989-01-01', 20),
('Sachin', '1991-02-05', 30);


CREATE TABLE employee_hoursworked (
    emp_name VARCHAR(20),
    work_date DATE,
    bill_hrs INT
);
INSERT INTO employee_hoursworked VALUES
('Sachin', '1990-07-01', 3),
('Sachin', '1990-08-01', 5),
('Sehwag', '1990-07-01', 2),
('Sachin', '1991-07-01', 4);

SELECT * FROM employee_billings;

emp_name,bill_date,bill_rate
Sachin,1990-01-01,25
Sehwag,1989-01-01,15
Dhoni,1989-01-01,20
Sachin,1991-02-05,30


In [0]:
%sql
SELECT * FROM employee_hoursworked;

emp_name,work_date,bill_hrs
Sachin,1990-07-01,3
Sachin,1990-08-01,5
Sehwag,1990-07-01,2
Sachin,1991-07-01,4


In [0]:
%sql
with cte as (
    select
        *,
        lead(to_date(dateadd(day, -1, bill_date)), 1, '9999-12-31') over(partition by emp_name order by bill_date) as next_bill_date
    from employee_billings
),
cte1 as(
    select 
        hw.*, 
        cte.bill_rate, 
        cte.bill_date, 
        cte.next_bill_date
    from cte
    inner join employee_hoursworked as hw 
    on cte.emp_name = hw.emp_name 
    where hw.work_date between cte.bill_date and cte.next_bill_date
)
select 
    cte1.emp_name,
    sum(cte1.bill_hrs * cte1.bill_rate) as total_bill
from cte1
group by cte1.emp_name;

emp_name,total_bill
Sachin,320
Sehwag,30


##### 06. Series of questions as we go along in it.

In [0]:
%sql
DROP TABLE IF EXISTS app_activity;
CREATE TABLE app_activity (
    user_id INT,
    event_name VARCHAR(20),
    event_date DATE,
    country VARCHAR(20)
);
INSERT INTO app_activity VALUES
(1, 'app-installed', '2022-01-01', 'India'),
(1, 'app-purchase',  '2022-01-02', 'India'),
(2, 'app-installed', '2022-01-01', 'USA'),
(3, 'app-installed', '2022-01-01', 'USA'),
(3, 'app-purchase',  '2022-01-03', 'USA'),
(4, 'app-installed', '2022-01-03', 'India'),
(4, 'app-purchase',  '2022-01-03', 'India'),
(5, 'app-installed', '2022-01-03', 'SL'),
(5, 'app-purchase',  '2022-01-03', 'SL'),
(6, 'app-installed', '2022-01-04', 'Pakistan'),
(6, 'app-purchase',  '2022-01-04', 'Pakistan');

SELECT * FROM app_activity ORDER BY user_id, event_date;

user_id,event_name,event_date,country
1,app-installed,2022-01-01,India
1,app-purchase,2022-01-02,India
2,app-installed,2022-01-01,USA
3,app-installed,2022-01-01,USA
3,app-purchase,2022-01-03,USA
4,app-installed,2022-01-03,India
4,app-purchase,2022-01-03,India
5,app-purchase,2022-01-03,SL
5,app-installed,2022-01-03,SL
6,app-installed,2022-01-04,Pakistan


In [0]:
%sql
--- Question1 - Find total active users each day

select
    event_date,
    count(distinct user_id) as total_active_users
from app_activity
group by event_date;

event_date,total_active_users
2022-01-01,3
2022-01-02,1
2022-01-03,3
2022-01-04,1


In [0]:
%sql
--- Question2 - Find total active users each week

SELECT
    DATE_PART('week', event_date) AS week_number,
    COUNT(DISTINCT user_id) AS total_active_users
FROM app_activity
GROUP BY DATE_PART('week', event_date)
ORDER BY week_number;

week_number,total_active_users
1,4
52,3


In [0]:
%sql
--- Question3: date wise total no of users who made the purchase same day they installed the app

with cte as(
    select
        event_date,
        user_id,
        case
            when count(distinct event_name) = 2 then user_id
            else null
        end as users_who_took_both
    from app_activity
    group by user_id, event_date
)
select
    event_date,
    count(users_who_took_both) as total_users
from cte
group by event_date;

event_date,total_users
2022-01-01,0
2022-01-02,0
2022-01-03,2
2022-01-04,1


In [0]:
%sql
--- Question 4: percentage of paid users in India, USA, and any other country should be tagged as others

with cte as (
select
    case
        when country in ('India', 'USA') then country
        else 'others'
    end as country,
    count(distinct user_id) as no_users
from app_activity
where event_name = "app-purchase"
group by
    case
        when country in ('India', 'USA') then country
        else 'others'
    end
)
select
    country,
    no_users,
    sum(no_users) over () as total_users,
    round(no_users/sum(no_users) over () * 100, 2) as percentage
from cte;

country,no_users,total_users,percentage
India,2,5,40.0
USA,1,5,20.0
others,2,5,40.0


In [0]:
%sql
--- Question 5: Among all the users who installed the app on a given day, how many did in app purchased on the very next day

with cte as (
select
    *,
    lead(event_name) over (partition by user_id order by event_date) as next_event,
    lead(event_date) over (partition by user_id order by event_date) as next_date
from app_activity
)
select
    event_date,
    sum(case
        when datediff(day, event_date, next_date) = 1 then 1
        else 0
    end) as paid
from cte
group by event_date

event_date,paid
2022-01-01,1
2022-01-02,0
2022-01-03,0
2022-01-04,0


#### ---- **Question 7** :----  3 or more consecutive empty seats
- ###### method 1 - lead lag
- ###### method 2 - advance aggregation using lag lead
- ###### method 3 - analytical row number function

In [0]:
%sql
DROP TABLE IF EXISTS bms;
CREATE TABLE bms (seat_no INT, is_empty VARCHAR(10));
INSERT INTO bms VALUES (1,'N'), (2,'Y'), (3,'N'), (4,'Y'), (5,'Y'), (6,'Y'), (7,'N'), (8,'Y'), (9,'Y'), (10,'Y'), (11,'Y'), (12,'N'), (13,'Y'), (14,'Y');

seat_no,is_empty
1,N
2,Y
3,N
4,Y
5,Y
6,Y
7,N
8,Y
9,Y
10,Y


In [0]:
%sql
-------------------------- Method 1 ------------------
select * from (
select *,
    lag(is_empty, 1) over(order by seat_no) as prev1,
    lag(is_empty, 2) over(order by seat_no) as prev2,
    lead(is_empty, 1) over(order by seat_no) as next1,
    lead(is_empty, 2) over(order by seat_no) as next2
from bms)
where is_empty = 'Y' and prev1 = 'Y' and prev2 = 'Y'
or (is_empty = 'Y' and prev1 = 'Y' and next1 = 'Y')
or (is_empty = 'Y' and next1 = 'Y' and next2 = 'Y') ;

seat_no,is_empty,prev1,prev2,next1,next2
4,Y,N,Y,Y,Y
5,Y,Y,N,Y,N
6,Y,Y,Y,N,Y
8,Y,N,Y,Y,Y
9,Y,Y,N,Y,Y
10,Y,Y,Y,Y,N
11,Y,Y,Y,N,Y


In [0]:
%sql
------------------------------ Method 2 ----------------------------------
with cte as(
select *,
    sum(
        case
            when is_empty = 'Y' then 1
            else 0
        end
        ) 
    over (order by seat_no rows between 2 preceding and current row) as prev2,
    sum(
        case
            when is_empty = 'Y' then 1
            else 0
        end
        ) 
    over (order by seat_no rows between 1 preceding and 1 following) as prev_next,
    sum(
        case
            when is_empty = 'Y' then 1
            else 0
        end
        ) 
    over (order by seat_no rows between current row and 2 following) as next2 
from bms
)
select * from cte
where prev2 = 3 or prev_next=3 or next2 = 3;

seat_no,is_empty,prev2,prev_next,next2
4,Y,2,2,3
5,Y,2,3,2
6,Y,3,2,2
8,Y,2,2,3
9,Y,2,3,3
10,Y,3,3,2
11,Y,3,2,2


In [0]:
%sql
-------------------------- Method 3 -------------------------
with cte as (
select *,
    row_number() over(order by seat_no) as rank,
    seat_no - row_number() over(order by seat_no) as diff
from bms
where is_empty = 'Y'
),
cte1 as (
select diff, count(*) from cte group by diff having count(*) >= 3)

select * from cte
where diff in (select diff from cte1)

seat_no,is_empty,rank,diff
4,Y,2,2
5,Y,3,2
6,Y,4,2
8,Y,5,3
9,Y,6,3
10,Y,7,3
11,Y,8,3


##### Question 8 : Find the missing Quarter for each Store

In [0]:
%sql
DROP TABLE IF EXISTS store_quarterly_sales;
CREATE TABLE store_quarterly_sales (
    Store VARCHAR(10),
    Quarter VARCHAR(10),
    Amount INT
);
INSERT INTO store_quarterly_sales (Store, Quarter, Amount) VALUES
    ('S1', 'Q1', 200),
    ('S1', 'Q2', 300),
    ('S1', 'Q4', 400),
    ('S2', 'Q1', 500),
    ('S2', 'Q3', 600),
    ('S2', 'Q4', 700),
    ('S3', 'Q1', 800),
    ('S3', 'Q2', 750),
    ('S3', 'Q3', 900);

num_affected_rows,num_inserted_rows
9,9


In [0]:
%sql
----------------------------- Method 1 ( Using normal substring/right ) --------------------------

-- select
--     Store,
--     concat("Q", int(10 - sum(substring(Quarter, 2)))) as missing_quarter
-- from store_quarterly_sales
-- group By Store;

select
    Store,
    concat("Q", int(10 - sum(right(Quarter, 1)))) as missing_quarter
from store_quarterly_sales
group By Store;

Store,missing_quarter
S1,Q3
S2,Q2
S3,Q4


In [0]:
%sql
---------------------------- Method 2 ( Using recurssive CTE ) -----------------------------
with recursive cte as (
    select distinct Store, 1 as q_no from store_quarterly_sales
    union all
    select Store, q_no+1 as q_no from cte where q_no < 4
),
cte1 as (
    select
        Store,
        concat("Q", q_no) as quarter
    from cte
    order by Store, q_no
)
select *
from cte1 as c
left join store_quarterly_sales as s
on c.Store = s.Store and c.quarter = s.Quarter
where s.Store is Null
order by c.Store;

Store,quarter,Store,Quarter,Amount
S1,Q3,null,null,null
S2,Q2,null,null,null
S3,Q4,null,null,null


##### Question 9 : Find students with same marks in Physics & Chemistry

In [0]:
%sql
drop table if exists exams;
create table exams (
    student_id int, 
    subject varchar(20), 
    marks int
);
delete from exams;
insert into exams values 
    (1,'Chemistry',91),(1,'Physics',91),(2,'Chemistry',80),(2,'Physics',90),(3,'Chemistry',80),(4,'Chemistry',71),(4,'Physics',54);

num_affected_rows,num_inserted_rows
7,7


In [0]:
%sql
------------------- Method 1 (using windows function lead) --------------------
with cte as (
select *,
    lead(marks) over (partition by student_id order by subject) as a
from exams
)
select student_id from cte
where marks = a;

student_id
1


In [0]:
%sql
--------------------------------  Method 2 (using GroupBy with Having clause)  -------------------------- 
select 
    student_id, 
    count(subject) 
from exams 
where subject in ("Chemistry", "Physics") 
group by student_id 
having count(subject) >= 2 and count(distinct marks) != 2; 

student_id,count(subject)
1,2


##### Question 10 - Find cities where the COVID cases are increasing continuously

In [0]:
%sql
drop table if exists covid;
create table covid(city varchar(50),days date,cases int);
delete from covid;
insert into covid values('DELHI','2022-01-01',100);
insert into covid values('DELHI','2022-01-02',200);
insert into covid values('DELHI','2022-01-03',300);
insert into covid values('MUMBAI','2022-01-01',100);
insert into covid values('MUMBAI','2022-01-02',100);
insert into covid values('MUMBAI','2022-01-03',300);
insert into covid values('CHENNAI','2022-01-01',100);
insert into covid values('CHENNAI','2022-01-02',200);
insert into covid values('CHENNAI','2022-01-03',150);
insert into covid values('BANGALORE','2022-01-01',100);
insert into covid values('BANGALORE','2022-01-02',300);
insert into covid values('BANGALORE','2022-01-03',200);
insert into covid values('BANGALORE','2022-01-04',400);

select * from covid;

city,days,cases
BANGALORE,2022-01-02,300
BANGALORE,2022-01-03,200
BANGALORE,2022-01-01,100
BANGALORE,2022-01-04,400
CHENNAI,2022-01-01,100
CHENNAI,2022-01-02,200
CHENNAI,2022-01-03,150
MUMBAI,2022-01-02,100
MUMBAI,2022-01-03,300
MUMBAI,2022-01-01,100


In [0]:
%sql
----------------- Method 1 ----------------------
with cte as (
select *,
    rank() over(partition by city order by days) as rank_days,
    rank() over(partition by city order by cases) as rank_cases,
    (rank() over(partition by city order by days)) - (rank() over(partition by city order by cases)) as diff
from covid order by city, days
)
select
    city
from cte
group by city
having count(distinct diff) = 1 and avg(diff) = 0;

city
DELHI


In [0]:
%sql
----------------------------- Method 2 -------------------------------
with cte as(
    select *,
        case 
            when cases < lead(cases,1,cases+1) over (partition by city order by days) then 1 else 0
        end as com 
    from covid)
select * from covid 
where city not in (
    select city from cte where com = 0
);

city,days,cases
DELHI,2022-01-03,300
DELHI,2022-01-02,200
DELHI,2022-01-01,100


##### Question 11 - Find companies who have atleast 2 users who speaks English & German both the languages

In [0]:
%sql
DROP TABLE IF EXISTS company_users;
CREATE TABLE company_users (
    company_id INT,
    user_id INT,
    language VARCHAR(20)
);
INSERT INTO company_users (company_id, user_id, language)
VALUES
    (1, 1, 'English'),
    (1, 1, 'German'),
    (1, 2, 'English'),
    (1, 3, 'German'),
    (1, 3, 'English'),
    (1, 4, 'English'),
    (2, 5, 'English'),
    (2, 5, 'German'),
    (2, 5, 'Spanish'),
    (2, 6, 'German'),
    (2, 6, 'Spanish'),
    (2, 7, 'English');

num_affected_rows,num_inserted_rows
12,12


In [0]:
%sql
select * from company_users;

company_id,user_id,language
1,1,English
1,1,German
1,2,English
1,3,German
1,3,English
1,4,English
2,5,English
2,5,German
2,5,Spanish
2,6,German


In [0]:
%sql
with cte as (
    select
        company_id,
        user_id
    from company_users
    where language in ('English','German')
    group by company_id, user_id
    having count(*) = 2
)
select company_id from cte
group by company_id
having count(user_id) = 2;

company_id
1


##### Question 12 - Find how many products falls under customer budget along with list of products

In [0]:
%sql
DROP TABLE IF EXISTS products;
CREATE TABLE products (
    product_id VARCHAR(20),
    cost INT
);
INSERT INTO products (product_id, cost)
VALUES
    ('P1', 200),
    ('P2', 300),
    ('P3', 500),
    ('P4', 800);

DROP TABLE IF EXISTS customer_budget;
CREATE TABLE customer_budget (
    customer_id INT,
    budget INT
);
INSERT INTO customer_budget (customer_id, budget)
VALUES
    (100, 400),
    (200, 800),
    (300, 1500);

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
SELECT * FROM products;

product_id,cost
P1,200
P2,300
P3,500
P4,800


In [0]:
%sql
SELECT * FROM customer_budget;

customer_id,budget
100,400
200,800
300,1500


In [0]:
%sql
with cte as (
select
    *,
    sum(cost) over (order by product_id) as r_cost
from products
)
select
    customer_id,
    budget,
    count(*) as num_products,
    string_agg(product_id, ',  ') as product_list
from customer_budget as c
left join cte
on cte.r_cost < c.budget
group by customer_id, budget;

customer_id,budget,num_products,product_list
100,400,1,P1
200,800,2,"P2, P1"
300,1500,3,"P3, P2, P1"


##### Question 13 - Find total number of messages exchanged between each person per day

In [0]:
%sql
drop table if exists subscriber;
CREATE TABLE subscriber (
 sms_date date ,
 sender varchar(20) ,
 receiver varchar(20) ,
 sms_no int
);
-- insert some values
INSERT INTO subscriber VALUES ('2020-4-1', 'Avinash', 'Vibhor',10);
INSERT INTO subscriber VALUES ('2020-4-1', 'Vibhor', 'Avinash',20);
INSERT INTO subscriber VALUES ('2020-4-1', 'Avinash', 'Pawan',30);
INSERT INTO subscriber VALUES ('2020-4-1', 'Pawan', 'Avinash',20);
INSERT INTO subscriber VALUES ('2020-4-1', 'Vibhor', 'Pawan',5);
INSERT INTO subscriber VALUES ('2020-4-1', 'Pawan', 'Vibhor',8);
INSERT INTO subscriber VALUES ('2020-4-1', 'Vibhor', 'Deepak',50);

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
with cte as (
select *,
    case
        when sender < receiver then sender else receiver 
    end as P1,
    case
        when sender > receiver then sender else receiver 
    end as P2
from subscriber
)
select
    sms_date,
    P1,
    P2,
    sum(sms_no) as total_sms_count
from cte
group by sms_date, P1, P2;

sms_date,P1,P2,total_sms_count
2020-04-01,Avinash,Vibhor,30
2020-04-01,Avinash,Pawan,50
2020-04-01,Deepak,Vibhor,50
2020-04-01,Pawan,Vibhor,13


##### Question 14 - Series of questions 

In [0]:
%sql
DROP TABLE IF EXISTS students;
CREATE TABLE students (
    studentid INT,
    studentname VARCHAR(255),
    subject VARCHAR(255),
    marks INT,
    testid INT,
    testdate DATE
);
INSERT INTO students (studentid, studentname, subject, marks, testid, testdate)
VALUES
    (2, 'Max Ruin', 'Subject1', 63, 1, '2022-01-02'),
    (3, 'Arnold', 'Subject1', 95, 1, '2022-01-02'),
    (4, 'Krish Star', 'Subject1', 61, 1, '2022-01-02'),
    (5, 'John Mike', 'Subject1', 91, 1, '2022-01-02'),
    (4, 'Krish Star', 'Subject2', 71, 1, '2022-01-02'),
    (3, 'Arnold', 'Subject2', 32, 1, '2022-01-02'),
    (5, 'John Mike', 'Subject2', 61, 2, '2022-11-02'),
    (1, 'John Deo', 'Subject2', 60, 1, '2022-01-02'),
    (2, 'Max Ruin', 'Subject2', 84, 1, '2022-01-02'),
    (2, 'Max Ruin', 'Subject3', 29, 3, '2022-01-03'),
    (5, 'John Mike', 'Subject3', 98, 2, '2022-11-02');
SELECT * FROM students ORDER BY studentid, subject;

studentid,studentname,subject,marks,testid,testdate
1,John Deo,Subject2,60,1,2022-01-02
2,Max Ruin,Subject1,63,1,2022-01-02
2,Max Ruin,Subject2,84,1,2022-01-02
2,Max Ruin,Subject3,29,3,2022-01-03
3,Arnold,Subject1,95,1,2022-01-02
3,Arnold,Subject2,32,1,2022-01-02
4,Krish Star,Subject1,61,1,2022-01-02
4,Krish Star,Subject2,71,1,2022-01-02
5,John Mike,Subject1,91,1,2022-01-02
5,John Mike,Subject2,61,2,2022-11-02


##### Question 14.01 - Get the list of students who scored above the average marks in each subject

In [0]:
%sql
with cte as (
select
    subject,
    avg(marks) as avg_marks
from students
group by subject
)
select
    students.subject,
    students.studentname,
    students.marks,
    cte.avg_marks
from cte
inner join students
on cte.subject = students.subject
where students.marks > cte.avg_marks;

subject,studentname,marks,avg_marks
Subject1,Arnold,95,77.5
Subject1,John Mike,91,77.5
Subject2,Krish Star,71,61.6
Subject2,Max Ruin,84,61.6
Subject3,John Mike,98,63.5


##### Question 14.02 - Get the percentage of students who score more than 90 in any subject amongst the total students

In [0]:
%sql
select
    100 * count(distinct(studentname))/(select count(distinct(studentname)) from students) as percentage
from students
where marks > 90;

percentage
40.0


##### Question 14.03 - Get the 2nd highest & 2nd lowest marks for each subject

In [0]:
%sql
with cte as (
select *,
    row_number() over (partition by subject order by marks desc) as high,
    row_number() over (partition by subject order by marks asc) as low
from students order by subject
)
select
    subject,
    max(case
        when high = 2 then marks end) as second_highest,
    max(case
        when low = 2 then marks end) as second_lowest
from cte
group by subject;

subject,second_highest,second_lowest
Subject1,91,63
Subject2,71,60
Subject3,29,98


##### Question 14.04 - For each student & test, identify if their marks increased or decreased from the previous test

In [0]:
%sql
with cte as (
select *,
    lag(marks, 1) over (partition by studentid order by testdate, subject) as prev_marks
from students order by studentid, testdate
)
select
    studentid,
    studentname,
    marks,
    prev_marks,
    testdate,
    case
        when prev_marks is null then 'no prev test'
        when prev_marks < marks then 'increased' else 'decreased'
    end as status
from cte;

studentid,studentname,marks,prev_marks,testdate,status
1,John Deo,60,null,2022-01-02,no prev test
2,Max Ruin,63,null,2022-01-02,no prev test
2,Max Ruin,84,63,2022-01-02,increased
2,Max Ruin,29,84,2022-01-03,decreased
3,Arnold,32,95,2022-01-02,decreased
3,Arnold,95,null,2022-01-02,no prev test
4,Krish Star,71,61,2022-01-02,increased
4,Krish Star,61,null,2022-01-02,no prev test
5,John Mike,91,null,2022-01-02,no prev test
5,John Mike,61,91,2022-11-02,decreased
